In [1]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
import re
import nltk
from nltk.corpus import stopwords
import pymorphy3
import pandas as pd

In [3]:
# Для русских текстов
morph = pymorphy3.MorphAnalyzer(lang='ru')
stop_words = set(stopwords.words('russian'))

def preprocess_text(text):
    # Удаление цензурных символов (****, ### и т.д.)
    text = re.sub(r'[*#]+', '', text)
    
    # Удаление спецсимволов, но сохранение основных знаков препинания
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s.,!?-]', ' ', text)
    
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Токенизация
    tokens = text.split()
    
    # Удаление стоп-слов и лемматизация
    cleaned_tokens = []
    for token in tokens:
        if token not in stop_words and len(token) > 2:
            lemma = morph.parse(token)[0].normal_form
            cleaned_tokens.append(lemma)
    
    return ' '.join(cleaned_tokens)

In [4]:
table_data = pd.read_excel("../../data_raw/DataSet_V49 (2).xlsx")

In [5]:
text_data = pd.read_csv("../../anonymized_parsed_data_analyzes.csv")

In [6]:
len(text_data)

14487

In [7]:
table_data.dropna(subset=['Смерть'], inplace=True)

In [8]:
len(table_data)

16613

In [9]:
df2_subset = table_data[['Файл(Анализы)', 'Смерть']]
df2_subset.rename(columns={'Файл(Анализы)': 'file_name', 'Смерть': 'label'}, inplace=True)

df2_subset["file_name"] = df2_subset["file_name"].astype(str)
# text_data['file_name'] = text_data['file_name'].apply(lambda x: str(int(x)) if pd.notna(x) else None)

# Делаем merge по идентификатору
df1 = text_data.merge(df2_subset, on='file_name', how='inner')

/tmp/ipykernel_2593233/3919788308.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset.rename(columns={'Файл(Анализы)': 'file_name', 'Смерть': 'label'}, inplace=True)
/tmp/ipykernel_2593233/3919788308.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset["file_name"] = df2_subset["file_name"].astype(str)


In [10]:
df1

,file_name,patient_id,anonymized_text_data,label
0,АсатрянРС_analyzes_0.html,7365.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
1,ПетровЕГ_analyzes_0.html,554.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
2,ДиктяревАВ_analyzes_0.html,10765.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
3,СвергунСЛ3428_analyzes_0.html,3428.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
4,КаргинСП5884_analyzes_0.html,5884.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
...,...,...,...,...
9477,КубрякАВ_analyzes_0.html,7018.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
9478,РассказовНТ2154_analyzes_0.html,2154.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
9479,КурашвилиГА_analyzes_0.html,10459.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
9480,МатисЮВ_analyzes_0.html,18882.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет


In [11]:
df1['anonymized_text_data'][0]

'Подождите, пожалуйста... Идет подготовка к выводу всех обследований и воздействий. Это может занять несколько минут. 100%\nУчреждение: Государственное бюджетное учреждение здравоохранения "Приморская краевая клиническая больница № 1"\nОбщий анализ крови_экспресс Выполнен 03.05.2019 в 16:36\nФ.И.О. [ФИО] ,\nДата рождения: 24.03.1953 , 68 лет , М\nОтделение Реанимации и интенсивной терапии РСЦ ,\nПалата № 2 Медицинская карта № 7365 /\nДиагноз ИБС. Нестабильная стенокардия умеренного риска. ПИКС. ОСН I ФК по Killip на фоне ХСН 2А ст., 2 ф.к.\nНаправил Врач-реаниматолог [ФИО]\nДата направления 03.05.2019 в 15:31 Дата взятия биоматериала: 03.05.2019 Время: 15:59\nИсследуемый компонент Результат Референсные значения и единица измерения Лейкоциты (WBC) 6.7 3.9-9 10^9/L Лимфоциты (Lym#) 0.8 1.2-3 % Лимфоциты (Lym%) 11.9 19-37 % Эритроциты (RBC) 3.85 3.9-5 10^12/L Гемоглобин (HGB) 116 120-160 g/L Гематокрит (HCT) 30.4 36-48 % Средний объем эритроцита (MCV) 79.1 80-100 fL Среднее содержание гем

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec, FastText

In [16]:
df1['anonymized_text_data'] = df1['anonymized_text_data'].apply(lambda x: preprocess_text(x))

In [17]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 3),
    analyzer='char_wb',
    sublinear_tf=True
)

X_texts = vectorizer.fit_transform(df1["anonymized_text_data"])

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X_texts, df1["label"], test_size=0.2, random_state=42, stratify=df1["label"]
)

In [19]:
text_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

text_model.fit(X_train, y_train)

y_pred_proba = text_model.predict_proba(X_test)[:, 1]
y_pred = text_model.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC на тесте: {auc:.4f}")

AUC на тесте: 0.9787


In [20]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          Да       0.49      0.89      0.63        70
         Нет       1.00      0.96      0.98      1827

    accuracy                           0.96      1897
   macro avg       0.74      0.93      0.81      1897
weighted avg       0.98      0.96      0.97      1897



In [28]:
count = 0
for i in sorted(y_pred):
    if i == 'Да':
        count += 1

print(count)

126
